In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
current_risk = pd.read_csv("../Dataset/student_risk_prediction.csv")

future_risk = pd.read_csv("../Dataset/future_risk_prediction.csv")

student_data = pd.read_csv("../Dataset/final_selected_dataset.csv")

In [3]:
print("Current Risk :", current_risk.shape)

print("Future Risk :", future_risk.shape)

print("Student Dataset :", student_data.shape)

Current Risk : (32593, 22)
Future Risk : (21180, 7)
Student Dataset : (32593, 16)


In [4]:
print(current_risk.columns.tolist())

print(future_risk.columns.tolist())

print(student_data.columns.tolist())

['Student ID', 'avg_activity_day', 'login_frequency', 'video_clicks', 'avg_quiz_score', 'avg_submission_day', 'assessments_completed', 'avg_assessment_weight', 'studied_credits', 'code_module', 'gender', 'code_presentation', 'highest_education', 'age_band', 'engagement_score', 'activity_per_login', 'quiz_completion', 'submission_consistency', 'Actual', 'Predicted', 'Risk Probability', 'Risk Level']
['Student ID', 'Prediction Week', 'Actual', 'Predicted', 'Risk Probability', 'Future Risk', 'Recommended Intervention']
['final_result', 'registration_days', 'avg_activity_day', 'login_frequency', 'video_clicks', 'avg_quiz_score', 'avg_submission_day', 'assessments_completed', 'avg_assessment_weight', 'studied_credits', 'code_module', 'gender', 'code_presentation', 'highest_education', 'age_band', 'dropout']


In [5]:
roadmap = current_risk.merge(

    future_risk,

    on="Student ID",

    how="left"

)

if "Student ID" not in roadmap.columns:

    roadmap.insert(

        0,

        "Student ID",

        range(1, len(roadmap)+1)

    )

In [6]:
roadmap["Future Risk"] = roadmap["Future Risk"].fillna(
    "Insufficient Weekly History"
)

roadmap["Recommended Intervention"] = roadmap[
    "Recommended Intervention"
].fillna(
    "Collect more weekly activity before predicting future risk."
)

In [7]:
current_risk = current_risk.loc[:, ~current_risk.columns.duplicated()]

future_risk = future_risk.loc[:, ~future_risk.columns.duplicated()]

student_data = student_data.loc[:, ~student_data.columns.duplicated()]

In [8]:
student_data = student_data.reset_index(drop=True)

student_data["Student ID"] = student_data.index + 1

In [9]:
current_risk = current_risk.reset_index(drop=True)

current_risk["Student ID"] = current_risk.index + 1

In [10]:
future_risk = future_risk.reset_index(drop=True)

future_risk["Student ID"] = future_risk.index + 1

In [11]:
current_risk = current_risk[
    [
        "Student ID",
        "Actual",
        "Predicted",
        "Risk Probability",
        "Risk Level"
    ]
]

future_risk = future_risk[
    [
        "Student ID",
        "Future Risk",
        "Recommended Intervention"
    ]
]

In [12]:
student_features = student_data[
    [
        "Student ID",
        "avg_activity_day",
        "login_frequency",
        "video_clicks",
        "avg_quiz_score",
        "avg_submission_day",
        "assessments_completed",
        "avg_assessment_weight",
        "studied_credits"
    ]
]

In [13]:
roadmap = current_risk.merge(

    future_risk,

    on="Student ID",

    how="left"

)

roadmap = roadmap.merge(

    student_features,

    on="Student ID",

    how="left"

)

print(roadmap.shape)

roadmap.head()

(32593, 15)


,Student ID,Actual,Predicted,Risk Probability,Risk Level,Future Risk,Recommended Intervention,avg_activity_day,login_frequency,video_clicks,avg_quiz_score,avg_submission_day,assessments_completed,avg_assessment_weight,studied_credits
0,1,0,0,0.018977,Low,Low,Continue Current Learning Path,102.132653,196.0,934.0,78.0,18.0,1.0,10.0,240
1,2,0,0,0.059905,Low,High,Immediate Faculty Intervention,86.993023,430.0,1435.0,70.0,22.0,1.0,10.0,60
2,3,1,1,0.691936,Medium,Low,Continue Current Learning Path,2.355263,76.0,281.0,0.0,0.0,0.0,0.0,60
3,4,0,0,0.007632,Low,Medium,Weekly Mentor Follow-up,106.147813,663.0,2158.0,72.0,17.0,1.0,10.0,60
4,5,0,0,0.042361,Low,Low,Continue Current Learning Path,91.934659,352.0,1034.0,69.0,26.0,1.0,10.0,60


In [14]:
# Fill numeric columns

numeric_columns = roadmap.select_dtypes(
    include=["number"]
).columns

roadmap[numeric_columns] = roadmap[numeric_columns].fillna(0)

# Fill string columns

string_columns = roadmap.select_dtypes(
    include=["object","string"]
).columns

roadmap[string_columns] = roadmap[string_columns].fillna("Not Available")

print(roadmap.isnull().sum())

Student ID                  0
Actual                      0
Predicted                   0
Risk Probability            0
Risk Level                  0
Future Risk                 0
Recommended Intervention    0
avg_activity_day            0
login_frequency             0
video_clicks                0
avg_quiz_score              0
avg_submission_day          0
assessments_completed       0
avg_assessment_weight       0
studied_credits             0
dtype: int64


In [15]:
learning_plan = []

for _, row in roadmap.iterrows():

    plan = []

    if row["Risk Level"] == "High":
        plan.append("Attend Faculty Mentoring Sessions")

    elif row["Risk Level"] == "Medium":
        plan.append("Weekly Mentor Follow-up")

    else:
        plan.append("Continue Current Progress")

    if row["video_clicks"] < 100:
        plan.append("Watch More Video Lectures")

    if row["login_frequency"] < 10:
        plan.append("Increase LMS Login Frequency")

    if row["avg_quiz_score"] < 50:
        plan.append("Practice Weekly Quizzes")

    if row["assessments_completed"] < 5:
        plan.append("Complete Pending Assessments")

    if row["avg_submission_day"] > 5:
        plan.append("Submit Assignments Earlier")

    learning_plan.append(" | ".join(plan))

roadmap["Learning Roadmap"] = learning_plan

In [16]:
print(roadmap.columns.tolist())

['Student ID', 'Actual', 'Predicted', 'Risk Probability', 'Risk Level', 'Future Risk', 'Recommended Intervention', 'avg_activity_day', 'login_frequency', 'video_clicks', 'avg_quiz_score', 'avg_submission_day', 'assessments_completed', 'avg_assessment_weight', 'studied_credits', 'Learning Roadmap']


In [17]:
roadmap[
    [
        "Student ID",
        "Actual",
        "Predicted",
        "Risk Probability",
        "Risk Level",
        "Future Risk",
        "Recommended Intervention",
        "Learning Roadmap"
    ]
].head(20)

,Student ID,Actual,Predicted,Risk Probability,Risk Level,Future Risk,Recommended Intervention,Learning Roadmap
0,1,0,0,0.018977,Low,Low,Continue Current Learning Path,Continue Current Progress | Complete Pending A...
1,2,0,0,0.059905,Low,High,Immediate Faculty Intervention,Continue Current Progress | Complete Pending A...
2,3,1,1,0.691936,Medium,Low,Continue Current Learning Path,Weekly Mentor Follow-up | Practice Weekly Quiz...
3,4,0,0,0.007632,Low,Medium,Weekly Mentor Follow-up,Continue Current Progress | Complete Pending A...
4,5,0,0,0.042361,Low,Low,Continue Current Learning Path,Continue Current Progress | Complete Pending A...
5,6,0,0,0.006106,Low,Low,Continue Current Learning Path,Continue Current Progress | Complete Pending A...
6,7,0,0,0.010655,Low,Medium,Weekly Mentor Follow-up,Continue Current Progress | Complete Pending A...
7,8,0,0,0.007261,Low,Low,Continue Current Learning Path,Continue Current Progress | Complete Pending A...
8,9,0,0,0.005220,Low,Low,Continue Current Learning Path,Continue Current Progress | Complete Pending A...
9,10,0,0,0.004757,Low,Low,Continue Current Learning Path,Continue Current Progress | Complete Pending A...


In [18]:
roadmap.to_csv(

    "../Dataset/personalized_learning_roadmap.csv",

    index=False

)

print("Personalized Learning Roadmap Saved Successfully")

Personalized Learning Roadmap Saved Successfully


In [19]:
print("=" * 60)

print("Total Students :", len(roadmap))

print()

print("Current Risk")

print(roadmap["Risk Level"].value_counts())

print()

print("Future Risk")

print(roadmap["Future Risk"].value_counts())

print("=" * 60)

Total Students : 32593

Current Risk
Risk Level
Low       18964
High       6931
Medium     6698
Name: count, dtype: int64

Future Risk
Future Risk
Low              13149
Not Available    11413
High              4020
Medium            4011
Name: count, dtype: int64


In [20]:
print(current_risk.shape)

print(future_risk.shape)

print(current_risk.columns.tolist())

print(future_risk.columns.tolist())

(32593, 5)
(21180, 3)
['Student ID', 'Actual', 'Predicted', 'Risk Probability', 'Risk Level']
['Student ID', 'Future Risk', 'Recommended Intervention']
